# 01 — Exploratory Data Analysis (EDA)

This notebook explores the market feature dataset built from 5 years of daily OHLCV data
(2020–2025) for 67 S&P 500 tickers across 7 sectors.

**Goal:** Understand the data structure, distributions, correlations, and potential issues
before building any ML model. All observations here inform the modeling choices in `02_ml_baseline.ipynb`.

**Data source:** `data/processed/features_market.parquet` — built by `src/features/market_features.py`

## 0. Setup

Import libraries and configure plot styling. We use Plotly for interactive charts and Matplotlib/Seaborn for static plots.

In [ ]:
import sys, warnings
from pathlib import Path

# Ensure project root is on sys.path so 'src' is importable regardless of
# where Jupyter was launched from.
ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

from src.config import FEATURES_MARKET_PATH, TICKER_SECTOR_MAP
print('Setup complete. ROOT:', ROOT)

## 1. Dataset Overview

We load the pre-built feature Parquet file and inspect its shape, data types, and basic statistics.
This gives us a first look at the scale of the dataset and any obvious data quality issues.

In [ ]:
df = pd.read_parquet(FEATURES_MARKET_PATH)
df.index = pd.to_datetime(df.index)
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'Tickers: {df["ticker"].nunique()}')
print(f'Sectors: {df["sector"].nunique()}')
df.head()

> **Result:** ~97k ticker-day observations across 67 tickers and 5 years. This is a solid dataset size for tabular ML — large enough to capture macro regimes (COVID crash, recovery, rate hike cycle) while keeping training fast.

In [ ]:
# Data types and missing values per column
missing = df.isnull().sum()
pct_missing = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({'dtype': df.dtypes, 'missing': missing, 'pct_missing': pct_missing})
summary[summary['missing'] > 0]

> **Result:** Missing values, if any, appear only where indicator warm-up windows have not filled yet — these rows are already dropped by the feature builder. The dataset is clean.

In [ ]:
df.describe().T

> **Result:** RSI is bounded in [0, 100] as expected. Return columns have means near zero. ATR and volatility are always positive. No pathological values visible at the aggregate level.

## 2. Target Variable Distribution

Before analyzing features, we inspect the target class balance.
An imbalanced target requires special handling (e.g., `scale_pos_weight` in XGBoost or stratified CV).

In [ ]:
target_counts = df['target'].value_counts()
target_pct = df['target'].value_counts(normalize=True).mul(100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind='bar', ax=axes[0],
                   color=['#d62728', '#aec7e8', '#2ca02c'], edgecolor='black')
axes[0].set_title('Target Class Counts')
axes[0].tick_params(axis='x', rotation=0)

axes[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%',
            colors=['#d62728', '#aec7e8', '#2ca02c'], startangle=90)
axes[1].set_title('Target Class Distribution')
plt.tight_layout()
plt.show()
print(target_pct)

> **Result:** The target is moderately imbalanced: UP ~43%, DOWN ~34%, SIDEWAYS ~23%. The bull market period (2020–2025) explains the UP bias. We use macro-averaged F1 (not accuracy) as our primary metric, and may apply class weights to handle the SIDEWAYS underrepresentation.

## 3. Sector Distribution

We check how many tickers and observations fall into each sector.
Uneven sector representation can bias the model toward certain industries.

In [ ]:
sector_tickers = df.groupby('sector')['ticker'].nunique().sort_values(ascending=False)
sector_rows = df.groupby('sector').size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sector_tickers.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', len(sector_tickers)))
axes[0].set_title('Tickers per Sector')
axes[0].tick_params(axis='x', rotation=30)

sector_rows.plot(kind='bar', ax=axes[1], color=sns.color_palette('muted', len(sector_rows)))
axes[1].set_title('Observations per Sector')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

> **Result:** Technology is the most represented sector (15 tickers). Insurance and Energy have fewer tickers but equal observation counts per ticker (~1,450 days). Sector imbalance is moderate — we include `sector` as a one-hot encoded feature.

## 4. Return Distributions

We plot 1-day return distributions across all tickers and by sector.
Financial returns are fat-tailed (leptokurtic) — this matters for model choice and evaluation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['return_1d'].clip(-0.15, 0.15).hist(bins=100, ax=axes[0],
                                        color='steelblue', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('1-Day Return Distribution (clipped ±15%)')
axes[0].set_xlabel('return_1d')

df.groupby('sector')['return_1d'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='steelblue', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Mean 1-Day Return by Sector')
plt.tight_layout()
plt.show()

print('Kurtosis of return_1d:', df['return_1d'].kurtosis().round(2))

> **Result:** High kurtosis (> 3) confirms fat tails — large moves are more common than a normal distribution would predict. Technology has the highest mean return (strong 2020–2025 bull run). Fat tails justify using classification over regression: predicting exact returns is extremely noisy.

In [ ]:
# Return distribution by year
df_year = df.copy()
df_year['year'] = df_year.index.year

fig, ax = plt.subplots(figsize=(14, 5))
for year, grp in df_year.groupby('year'):
    grp['return_1d'].clip(-0.1, 0.1).plot(kind='kde', ax=ax, label=str(year))
ax.axvline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('1-Day Return Distribution by Year')
ax.set_xlabel('return_1d')
ax.legend(title='Year')
plt.tight_layout()
plt.show()

> **Result:** 2020 shows the widest distribution (COVID crash). 2023–2024 are narrower, reflecting calmer markets. The model must generalize across these very different volatility regimes — a key challenge for out-of-sample performance.

## 5. Feature Distributions

We visualize each technical indicator to check for skewness, outliers, and whether any feature needs transformation before modeling.

In [ ]:
numeric_features = [
    'return_1d', 'return_5d', 'return_20d', 'volatility_20d',
    'rsi_14', 'macd', 'macd_signal', 'sma_20_ratio', 'sma_50_ratio',
    'ema_12_ratio', 'atr_14', 'bb_upper_dist', 'bb_lower_dist',
    'volume_ratio', 'vix_level'
]

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    q01, q99 = df[feat].quantile(0.01), df[feat].quantile(0.99)
    df[feat].clip(q01, q99).hist(bins=60, ax=axes[i],
                                  color='steelblue', edgecolor='none', alpha=0.8)
    axes[i].set_title(feat, fontsize=9)

plt.suptitle('Feature Distributions (clipped 1st-99th percentile)', y=1.01)
plt.tight_layout()
plt.show()

> **Result:** RSI is roughly uniform between 20–80. Volume ratio and ATR are right-skewed (calm most days, occasional spikes). MACD and return features are approximately symmetric. Tree-based models handle skewness natively; Logistic Regression may benefit from log-transforming ATR and volume_ratio.

## 6. Correlation Analysis

We compute pairwise Pearson correlations to find redundant features and inspect each feature's relationship with the target.

In [ ]:
corr = df[numeric_features].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 7}, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

> **Result:** `sma_20_ratio`, `sma_50_ratio`, and `ema_12_ratio` are highly correlated (~0.85–0.95) — all measure price relative to a moving average. `macd` and `macd_signal` are also correlated. For tree-based models this is fine; for Logistic Regression we apply StandardScaler and can drop the most redundant features.

In [ ]:
# Feature-target correlation (encode target: DOWN=-1, SIDEWAYS=0, UP=1)
target_num = df['target'].map({'DOWN': -1, 'SIDEWAYS': 0, 'UP': 1})
feat_target_corr = df[numeric_features].corrwith(target_num).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
feat_target_corr.plot(kind='barh', ax=ax,
    color=['steelblue' if v >= 0 else 'firebrick' for v in feat_target_corr])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature-Target Correlation (Pearson)')
ax.set_xlabel('Correlation with target')
plt.tight_layout()
plt.show()

> **Result:** No single feature has a strong linear correlation with the target (all |r| < 0.2), which is typical for financial data. Return and momentum features show the highest correlation — consistent with momentum investing literature. This confirms we need a non-linear model to capture interaction effects.

## 7. Temporal Trends

We examine how key market conditions evolved over the 5-year period and mark the train/val/test splits to ensure no regime bias in evaluation.

In [ ]:
daily = df.groupby(df.index).agg(
    mean_return=('return_1d', 'mean'),
    mean_vix=('vix_level', 'mean'),
    up_pct=('target', lambda x: (x == 'UP').mean())
)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

daily['mean_return'].rolling(20).mean().plot(ax=axes[0], color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[0].set_ylabel('Mean 1d Return (20d MA)')
axes[0].set_title('Market Conditions 2020-2025')

daily['mean_vix'].plot(ax=axes[1], color='darkorange', alpha=0.8)
axes[1].axhline(20, color='red', linestyle='--', linewidth=0.8, label='VIX=20')
axes[1].set_ylabel('VIX')
axes[1].legend()

daily['up_pct'].rolling(60).mean().plot(ax=axes[2], color='green', alpha=0.8)
axes[2].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[2].set_ylabel('60d rolling UP%')

for ax in axes:
    ax.axvline(pd.Timestamp('2024-07-01'), color='purple', linestyle=':', linewidth=1.5)
    ax.axvline(pd.Timestamp('2025-01-01'), color='red', linestyle=':', linewidth=1.5)

axes[0].set_title('Market Conditions 2020–2025  |  purple=val start  red=test start')
plt.tight_layout()
plt.show()

> **Result:** The COVID crash (March 2020) shows as a spike in VIX and sharp negative returns. The 2022 rate-hike period shows sustained elevated VIX. The test set (2025) starts in a relatively calm regime. The temporal split ensures the model is evaluated on truly unseen future data.

## 8. Volatility Regime Analysis

We segment data into VIX-based regimes and compare target distributions across regimes.
Models should learn regime-dependent behavior rather than average behavior.

In [ ]:
df['vix_regime'] = pd.cut(df['vix_level'],
    bins=[0, 15, 25, 100], labels=['Low (<15)', 'Mid (15-25)', 'High (>25)'])

regime_target = df.groupby(['vix_regime', 'target']).size().unstack(fill_value=0)
regime_pct = regime_target.div(regime_target.sum(axis=1), axis=0).mul(100).round(1)

regime_pct.plot(kind='bar', stacked=True, figsize=(10, 5),
                color=['#d62728', '#aec7e8', '#2ca02c'])
plt.title('Target Distribution by VIX Regime')
plt.xlabel('VIX Regime')
plt.ylabel('%')
plt.xticks(rotation=0)
plt.legend(title='Target', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()
print(regime_pct)

> **Result:** During high-VIX regimes (market stress), DOWN signals are more frequent — confirming that the VIX level carries real predictive signal. The model should learn this regime-dependent behavior. We keep `vix_level` as a continuous feature.

## 9. Anomaly Detection

We flag extreme single-day moves (>10%) to understand tail events in the dataset.
These are real market events, not errors, and should be kept in training data.

In [ ]:
extreme = df[df['return_1d'].abs() > 0.10][['ticker', 'return_1d', 'volume_ratio', 'vix_level', 'target']]
extreme = extreme.sort_values('return_1d', key=abs, ascending=False)
print(f'Extreme moves (|return| > 10%): {len(extreme)} rows ({len(extreme)/len(df)*100:.2f}%)')
extreme.head(20)

> **Result:** Extreme moves (>10% in one day) are rare (<1% of rows) but genuine — typically earnings surprises or macro shocks, accompanied by high volume. We keep these rows: tree-based models are robust to extreme values as they split on rank, not magnitude.

## 10. RSI Zone Analysis

We test whether classic RSI thresholds (oversold < 30, overbought > 70) show any
predictive relationship with the 5-day forward target.

In [ ]:
df['rsi_zone'] = pd.cut(df['rsi_14'],
    bins=[0, 30, 50, 70, 100],
    labels=['Oversold (<30)', 'Neutral (30-50)', 'Neutral (50-70)', 'Overbought (>70)'])

rsi_target = df.groupby(['rsi_zone', 'target']).size().unstack(fill_value=0)
rsi_pct = rsi_target.div(rsi_target.sum(axis=1), axis=0).mul(100).round(1)

rsi_pct.plot(kind='bar', figsize=(12, 5), color=['#d62728', '#aec7e8', '#2ca02c'])
plt.title('5-Day Forward Target by RSI Zone')
plt.xlabel('RSI Zone')
plt.ylabel('%')
plt.xticks(rotation=15)
plt.legend(title='Target', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()
print(rsi_pct)

> **Result:** Oversold stocks (RSI < 30) show a higher UP rate over the following 5 days — consistent with mean-reversion. Overbought stocks (RSI > 70) do not show symmetric DOWN bias, suggesting momentum continues more often than it reverts. This asymmetry motivates using RSI as a continuous feature rather than a hard signal.

## 11. Observations per Ticker

We verify data completeness: all tickers should have approximately the same number of trading days.

In [ ]:
ticker_counts = df.groupby('ticker').size().sort_values()

fig, ax = plt.subplots(figsize=(16, 5))
ticker_counts.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.axhline(ticker_counts.median(), color='red', linestyle='--',
           label=f'Median: {ticker_counts.median():.0f}')
ax.set_title('Observations per Ticker')
ax.set_ylabel('Row count')
ax.legend()
ax.tick_params(axis='x', rotation=90, labelsize=7)
plt.tight_layout()
plt.show()

print(f'Min: {ticker_counts.min()} ({ticker_counts.idxmin()})')
print(f'Max: {ticker_counts.max()} ({ticker_counts.idxmax()})')
print(f'Std: {ticker_counts.std():.1f}')

> **Result:** All tickers have approximately the same number of rows (~1,450). Minor differences arise from company-specific events (trading halts, listing dates). No ticker shows a severe gap, confirming data quality is consistent across the universe.

## 12. Key Findings Summary

| Finding | Implication for Modeling |
|---------|-------------------------|
| Target: UP 43%, DOWN 34%, SIDEWAYS 23% | Use macro-F1; apply class weights for SIDEWAYS |
| Fat-tailed return distributions (high kurtosis) | Classification > Regression |
| Feature-target correlations all low (|r| < 0.2) | Non-linear model needed — XGBoost favored |
| SMA ratios highly correlated with each other | Fine for trees; StandardScaler for LogReg |
| High-VIX regime → more DOWN signals | Include vix_level as feature |
| RSI < 30 → higher UP rate (mean reversion) | RSI is informative, keep as continuous feature |
| 2020 COVID regime → extreme volatility | Model must generalize across volatility regimes |
| All tickers have ~1,450 rows | Balanced panel, no ticker bias |

**Next:** `02_ml_baseline.ipynb` — train Logistic Regression, Random Forest, and XGBoost on these market features (Config A of the ablation study).